# NYC Motor Vehicle Collisions: Econometric & Machine Learning Fatality Analysis
**Course:** CIS 9660 — Data Mining  
**Authors:** Daniel Ohebshalom, Dennis Prasad, Dylan Lee  

---

## Executive Summary & Hypotheses
This notebook investigates whether vehicle manufacturer—specifically **Ford**—exerts a statistically significant impact on motor vehicle crash fatalities when controlling for driver demographics, vehicle deterioration (age), vehicle body type, and environmental timing (time of day and season).

### Hypotheses Tested
* **Hypothesis 1 (Brand Liability):**
  * $H_0$: Ford vehicles are a significant cause of fatalities in accidents, more so than other environmental factors.
  * $H_1$: Ford vehicles have a higher fatal crash rate than other vehicle makes.
* **Hypothesis 2 (Driver Demographics):**
  * $H_0$: Male drivers and Female drivers have the same fatal crash rates according to the data.
  * $H_1$: Male drivers lead to more fatalities compared to Female drivers.
* **Hypothesis 3 (Vehicle Age & Structural Decay):**
  * $H_0$: Vehicle age does not generally affect the chances of a fatal crash.
  * $H_1$: The older a car is, the more likely that factor holds major significance in a fatal crash occurring.
* **Hypothesis 4 (Environmental Timing — Time of Day & Seasonality):**
  * $H_0$: The time of day and time of year play a significant role in whether a crash is fatal.
  * $H_1$: The later it is in the year, and the later it is in the day, the more likely a crash is fatal.

### Experimental Architecture
To demonstrate the iterative data mining workflow that led to our final conclusions, this notebook is organized into **8 sequential phases**:
1. **Environment Setup & Data Ingestion Pipeline:** Demonstrates why a naive `.dropna()` on the raw 89,102-row dataset fails (`0` rows retained) due to 4 sparse columns (`>50%` missing), and how pruning those columns first preserves **39,793** complete crash records.
2. **Data Profiling & Outlier Discovery Dashboard:** Identifies extreme numerical typos (`VEHICLE_YEAR = 9999`, `VEHICLE_AGE = -7973`, `VEHICLE_OCCUPANTS = 327`) via descriptive statistics and a 2x2 Plotly Boxplot Dashboard.
3. **Exploratory Data Analysis (EDA) & Demographic Crosstabs:** Evaluates Ford's 3,823 crashes across time of day and gender, compares male collision skews across the Top 5 brands, and maps crash frequency/lethality across months and hours.
4. **Experiment 1 — Contributing Factor Screening (OLS HC0 & Melted Logit):** Runs an initial Linear Probability Model (`OLS` with White's `HC0` robust standard errors) and uses `pd.melt()` to stack `CONTRIBUTING_FACTOR_1` and `CONTRIBUTING_FACTOR_2` to test the Top 5 and Top 20 deadliest causes.
5. **Experiment 2 — Brand Mortality & Perfect Separation Diagnostics:** Computes fatality rates per 10,000 crashes across top makes and diagnoses **Quasi-Separation / Perfect Separation** caused by `NISSAN` having `0` fatalities in Maximum Likelihood Estimation (MLE).
6. **Experiment 3 — Modular Block-Wise Hypothesis Sub-Models (Models 1–4):** Tests `FORD` against isolated confounder blocks (Gender, Time of Day, Vehicle Type, Seasonality).
7. **Experiment 4 — Outlier Sanitization, Master Unified Logit & Male Subsample:** Evaluates the 9-predictor intermediate model ($p = 0.1168$) vs. the **12-predictor Master Logit Model** ($N = 39,689$, $p = 0.0002980$) and the **Male-Only Subsample Model** ($N = 28,585$).
8. **Experiment 5 — Scikit-Learn Regularized Logistic Regression & Visual Suite:** Benchmarks our econometric findings against an industry-standard `scikit-learn` $L_2$-penalized `Pipeline` (resolving MLE quasi-separation in `SPRING`) and exports publication-ready charts.

In [ ]:
# ==============================================================================
# SECTION 1: ENVIRONMENT SETUP & IMPORTS
# ==============================================================================
import calendar
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

try:
    from IPython.display import display
except ImportError:
    display = print

# Scikit-Learn Conventions for Machine Learning Validation
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# Output directory for saved charts and HTML dashboards
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Environment initialized successfully.")

## Phase 1: Smart Data Ingestion & Missingness Pruning
When working with the raw NYPD crash dataset (`Motor_Vehicle_Collisions_-_Crashes.csv`, $N = 89,102$), a naive `df.dropna()` removes **100% of the dataset** (`0` rows remaining) because four auxiliary columns (`VEHICLE_MODEL`, `VEHICLE_DAMAGE_2`, `VEHICLE_DAMAGE_3`, `PUBLIC_PROPERTY_DAMAGE_TYPE`) are over 50% empty.

Below, our ingestion pipeline automatically locates the dataset (checking both the current repository folder and `/Users/danieloheb/Desktop/CIS 9660`), demonstrates the column-pruning logic if the raw file is available, and loads the cleaned 39,793-row dataset (`Cleaned_Crashes_Time_Categorized.csv`).

In [ ]:
# ==============================================================================
# PHASE 1: SMART PATH RESOLUTION & SPARSITY CLEANING PIPELINE
# ==============================================================================
SEARCH_DIRS = [Path("."), Path("/Users/danieloheb/Desktop/CIS 9660")]

def find_file(filename: str) -> Path | None:
    """Search current directory and fallback project directory for a dataset."""
    for directory in SEARCH_DIRS:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return None

def categorize_time(time_str: str) -> str:
    """Categorize HH:MM crash time into Morning, Afternoon, Peak Hours, or Night."""
    try:
        hour = int(str(time_str).split(":")[0])
        if 6 <= hour < 12:
            return "Morning"
        elif 12 <= hour < 16:
            return "Afternoon"
        elif 16 <= hour < 19:
            return "Peak Hours"
        else:
            return "Night"
    except Exception:
        return "Unknown"

raw_csv_path = find_file("Motor_Vehicle_Collisions_-_Crashes.csv")
clean_csv_path = find_file("Cleaned_Crashes_Time_Categorized.csv")

# If raw dataset is present, demonstrate the exact cleaning & column-dropping experiment
if raw_csv_path is not None:
    df_raw = pd.read_csv(raw_csv_path, low_memory=False)
    print(f"Original raw rows: {len(df_raw):,}")
    print(f"Rows after naive complete dropna(): {len(df_raw.dropna()):,}")

    cols_to_drop = [col for col in df_raw.columns if df_raw[col].isnull().mean() > 0.50]
    df_practical = df_raw.drop(columns=cols_to_drop).dropna().copy()
    df_practical["Crash_Time_Occur"] = df_practical["CRASH_TIME"].apply(categorize_time)

    print(f"\nColumns dropped due to >50% missing: {cols_to_drop}")
    print(f"Rows after dropping sparse columns and then dropna(): {len(df_practical):,}")

    if clean_csv_path is None:
        clean_csv_path = Path("Cleaned_Crashes_Time_Categorized.csv")
        df_practical.to_csv(clean_csv_path, index=False)
        print(f"Saved cleaned dataset to {clean_csv_path}")

# Load the primary working dataset (Cleaned_Crashes_Time_Categorized.csv)
if clean_csv_path is not None:
    my_data = pd.read_csv(clean_csv_path, low_memory=False)
    if "Crash_Time_Occur" not in my_data.columns:
        my_data["Crash_Time_Occur"] = my_data["CRASH_TIME"].apply(categorize_time)
    if "VEHICLE_AGE" not in my_data.columns and "VEHICLE_YEAR" in my_data.columns:
        my_data["VEHICLE_AGE"] = 2026 - pd.to_numeric(my_data["VEHICLE_YEAR"], errors="coerce")
    print(f"\nLoaded working dataset: {clean_csv_path} ({my_data.shape[0]:,} rows x {my_data.shape[1]} columns)")
else:
    raise FileNotFoundError(
        "Could not find 'Cleaned_Crashes_Time_Categorized.csv' or 'Motor_Vehicle_Collisions_-_Crashes.csv'. "
        "Please place the CSV file in the same directory as this notebook."
    )

print("\nDistribution of Crash_Time_Occur:")
print(my_data["Crash_Time_Occur"].value_counts())
display(my_data.head())

## Phase 2: Initial Data Profiling & Outlier Discovery Dashboard
Before running our final econometric models, we inspect the summary statistics of our numerical features. Notice two critical data-entry anomalies in the raw/initial table:
1. `VEHICLE_YEAR` has a maximum value of **`9999.0`**, which produces an impossible `VEHICLE_AGE` minimum of **`-7973.0`**.
2. `VEHICLE_OCCUPANTS` has a maximum value of **`327.0`** (and `999.0` in the uncleaned file).

We visualize these anomalies using a **2x2 Plotly Boxplot Outlier Dashboard** with independent y-axis scales.

In [ ]:
# ==============================================================================
# PHASE 2: SUMMARY STATISTICS & 2x2 OUTLIER DETECTION DASHBOARD
# ==============================================================================
print("--- Initial Summary Statistics (Before Outlier Sanitization) ---")
display(my_data.describe())

num_cols = ["VEHICLE_OCCUPANTS", "NUMBER OF PERSONS INJURED", "NUMBER OF PERSONS KILLED", "VEHICLE_YEAR"]
df_box = my_data.copy()
for col in num_cols:
    if col in df_box.columns:
        df_box[col] = pd.to_numeric(df_box[col], errors="coerce")

fig_outliers = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Vehicle Occupants",
        "Number of Persons Injured",
        "Number of Persons Killed",
        "Vehicle Year",
    ],
)

positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for i, col in enumerate(num_cols):
    if col in df_box.columns:
        r, c = positions[i]
        fig_outliers.add_trace(
            go.Box(
                y=df_box[col].dropna(),
                name=col,
                boxpoints="outliers",
                marker_color="rgb(8,81,156)",
            ),
            row=r,
            col=c,
        )

fig_outliers.update_layout(
    height=700,
    width=950,
    title_text="Outlier Detection Dashboard (Independent Y-Axis Scales)",
    showlegend=False,
    template="plotly_white",
)
fig_outliers.write_html(OUTPUT_DIR / "all_variables_outliers_subplots.html")
fig_outliers.show()

## Phase 3: Exploratory Data Analysis (EDA) — Ford, Gender & Temporal Trends
To test our hypotheses from a descriptive perspective before controlling for confounders in a Logit regression, we examine:
1. **Ford Crash Breakdown ($N = 3,823$):** Crash frequency segmented by Time of Day (`Morning`, `Afternoon`, `Night`, `Peak Hours`) and Driver Gender (`M`, `F`, `U`).
2. **Brand-by-Gender Collision Skew:** Comparing the percentage of male collisions across the Top 5 vehicle brands (`Ford`, `Toyota`, `Honda`, `Nissan`, `Chevrolet`).While males account for the majority of collisions across all makes, Ford (`~81.7%`) and Chevrolet (`~77.8%`) skew much more heavily male than Honda (`~63.4%`) or Nissan (`~62.8%`).

In [ ]:
# ==============================================================================
# PHASE 3: FORD TIME OF DAY VS. GENDER & CROSS-BRAND GENDER SKEW
# ==============================================================================
df_eda = my_data.copy()
df_eda["IS_FORD"] = df_eda["VEHICLE_MAKE"].astype(str).str.upper().str.contains("FORD")
ford_crashes = df_eda[df_eda["IS_FORD"]].copy()

print(f"Number of Ford vehicle crashes:\n{len(ford_crashes)}")
print("=" * 35)
print("TIME OF DAY ANALYSIS")
print("=" * 35)
print("Crash Counts by Time of Day:")
print(ford_crashes["Crash_Time_Occur"].value_counts())

if "DRIVER_SEX" in ford_crashes.columns:
    print("=" * 35)
    print("GENDER ANALYSIS")
    print("=" * 35)
    print("Driver Gender Counts:")
    print(ford_crashes["DRIVER_SEX"].value_counts())

    print("=" * 35)
    print("GENDER VS TIME OF DAY")
    print("=" * 35)
    ct_ford_gender_time = pd.crosstab(ford_crashes["Crash_Time_Occur"], ford_crashes["DRIVER_SEX"])
    print(ct_ford_gender_time)

    print("=" * 35)
    print("SUMMARY")
    print("=" * 35)
    print(f"Total Ford Crashes: {len(ford_crashes)}")
    if not ford_crashes["Crash_Time_Occur"].empty:
        print(f"Most Common Crash Time:\n{ford_crashes['Crash_Time_Occur'].mode()[0]}")
    if not ford_crashes["DRIVER_SEX"].dropna().empty:
        print(f"Most Common Driver Gender:\n{ford_crashes['DRIVER_SEX'].mode()[0]}")

    # Plot Ford Vehicle Crashes: Time of Day vs Gender
    ax = ct_ford_gender_time.plot(kind="bar", figsize=(10, 6), width=0.75)
    plt.title("Ford Vehicle Crashes: Time of Day vs Gender", fontsize=14, fontweight="bold")
    plt.xlabel("Time of Day", fontsize=12)
    plt.ylabel("Number of Crashes", fontsize=12)
    plt.xticks(rotation=0)
    plt.legend(title="Gender")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ford_crashes_time_vs_gender.png", dpi=300)
    plt.show()

# Cross-Brand Male Collision Table (uses raw dataset if available to show full 89k counts, else cleaned)
df_brand_source = df_raw.copy() if raw_csv_path is not None else my_data.copy()

def map_brand_name(make: str) -> str:
    m = str(make).upper().strip()
    if "FORD" in m:
        return "Ford"
    if "TOYT" in m or "TOYOTA" in m:
        return "Toyota"
    if "HOND" in m:
        return "Honda"
    if "NISS" in m:
        return "Nissan"
    if "CHEV" in m:
        return "Chevrolet"
    return "Other"

df_brand_source["Brand_Clean"] = df_brand_source["VEHICLE_MAKE"].apply(map_brand_name)
df_top5_sex = df_brand_source[
    df_brand_source["Brand_Clean"].isin(["Ford", "Toyota", "Honda", "Nissan", "Chevrolet"])
    & df_brand_source["DRIVER_SEX"].isin(["M", "F", "U"])
]
brand_gender_table = pd.crosstab(df_top5_sex["Brand_Clean"], df_top5_sex["DRIVER_SEX"])
for g in ["M", "F", "U"]:
    if g not in brand_gender_table.columns:
        brand_gender_table[g] = 0
brand_gender_table = brand_gender_table[["M", "F", "U"]].rename(
    columns={"M": "Male (M)", "F": "Female (F)", "U": "Unknown (U)"}
)
brand_gender_table["Total"] = brand_gender_table.sum(axis=1)
brand_gender_table["% Male"] = (brand_gender_table["Male (M)"] / brand_gender_table["Total"] * 100).round(1)

print("\n--- Relationship Between DRIVER_SEX & VEHICLE_MAKE (% Male Collisions) ---")
display(brand_gender_table.loc[[b for b in ["Ford", "Toyota", "Honda", "Nissan", "Chevrolet"] if b in brand_gender_table.index]])

## Phase 4: Experimental Modeling Progression — Contributing Factors & Brand Separation Diagnostics

### Experiment 1: Linear Probability Model (OLS `HC0`) & Melted Multi-Factor Logit
In our first modeling iteration, we explored whether police-recorded crash causes (`CONTRIBUTING_FACTOR_1` and `CONTRIBUTING_FACTOR_2`) directly predict fatalities:
1. **Experiment 1A (OLS with White's `HC0` Robust Standard Errors):** Fits a baseline Linear Probability Model on `CONTRIBUTING_FACTOR_1` dummy variables and plots `NUMBER OF PERSONS KILLED` against `VEHICLE_YEAR` (which visually exposed the `9999` year typo).
2. **Experiment 1B (Multi-Column Melting):** Because a crash can record a lethal factor in either `CONTRIBUTING_FACTOR_1` or `CONTRIBUTING_FACTOR_2`, we use `pd.melt()` to stack both columns, drop `'Unspecified'` entries, deduplicate by `UNIQUE_ID`, and run a Logistic Regression on the **Top 5** deadliest contributing factors.

In [ ]:
# ==============================================================================
# EXPERIMENT 1A: INITIAL OLS REGRESSION (HC0 ROBUST SE) & SCATTER DIAGNOSTIC
# ==============================================================================
df_exp1 = my_data.copy()
y_ols = pd.to_numeric(df_exp1["NUMBER OF PERSONS KILLED"], errors="coerce").fillna(0)
X_ols = pd.get_dummies(df_exp1["CONTRIBUTING_FACTOR_1"], drop_first=True, dtype=int)
X_ols = sm.add_constant(X_ols)

model1_ols = sm.OLS(y_ols, X_ols).fit(cov_type="HC0")
df_exp1["predicted1"] = model1_ols.predict(X_ols)
print(f"Experiment 1A — OLS (HC0) fitted across {X_ols.shape[1]-1} contributing factors | R-squared: {model1_ols.rsquared:.4f}")

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_exp1, x="NUMBER OF PERSONS KILLED", y="VEHICLE_YEAR")
plt.title("Diagnostic Scatterplot: Persons Killed vs. Raw Vehicle Year (Exposing 9999 Outlier)")
plt.tight_layout()
plt.show()

# ==============================================================================
# EXPERIMENT 1B: MELTING CONTRIBUTING_FACTOR_1 & CONTRIBUTING_FACTOR_2 FOR LOGIT
# ==============================================================================
df_melt_base = my_data.dropna(subset=["NUMBER OF PERSONS KILLED"]).copy()
df_melt_base["IS_KILLED"] = (pd.to_numeric(df_melt_base["NUMBER OF PERSONS KILLED"], errors="coerce") > 0).astype(int)

melted = df_melt_base.melt(
    id_vars=["UNIQUE_ID", "NUMBER OF PERSONS KILLED"],
    value_vars=["CONTRIBUTING_FACTOR_1", "CONTRIBUTING_FACTOR_2"],
    value_name="FACTOR",
)
melted = melted.dropna(subset=["FACTOR"])
melted = melted[melted["FACTOR"] != "Unspecified"]
melted = melted.drop_duplicates(subset=["UNIQUE_ID", "FACTOR"])

deadly_factors_top5 = (
    melted.groupby("FACTOR")["NUMBER OF PERSONS KILLED"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print(f"\nTop 5 Deadliest Contributing Factors (across both factor columns):\n{deadly_factors_top5}")

for factor in deadly_factors_top5:
    df_melt_base[factor] = (
        (df_melt_base["CONTRIBUTING_FACTOR_1"] == factor)
        | (df_melt_base["CONTRIBUTING_FACTOR_2"] == factor)
    ).astype(int)

# Filter out crashes where both factors are Unspecified or NaN to establish a fair baseline
known_crashes = df_melt_base[
    (df_melt_base["CONTRIBUTING_FACTOR_1"].notna() & (df_melt_base["CONTRIBUTING_FACTOR_1"] != "Unspecified"))
    | (df_melt_base["CONTRIBUTING_FACTOR_2"].notna() & (df_melt_base["CONTRIBUTING_FACTOR_2"] != "Unspecified"))
].copy()

y_logit_f = known_crashes["IS_KILLED"]
X_logit_f = sm.add_constant(known_crashes[deadly_factors_top5])

try:
    logit_factors_model = sm.Logit(y_logit_f, X_logit_f).fit(disp=False)
    print("\n=== LOGISTIC REGRESSION (Top 5 Contributing Factors Across Both Columns) ===")
    print(logit_factors_model.summary())
except Exception as e:
    print(f"Regularized fallback triggered for Factor Logit: {e}")
    logit_factors_model = sm.Logit(y_logit_f, X_logit_f).fit_regularized(disp=False)
    print(logit_factors_model.summary())

### Experiment 2: Brand Mortality Rates & Perfect Separation Diagnostics
Next, we isolated the Top 5 vehicle makes (`TOYT`, `HOND`, `NISS`, `FORD`, `CHEV`) to test whether Ford exhibits a higher fatality rate per 10,000 crashes.

**Key Methodological Discovery (Perfect Separation in MLE):**  
When we expanded the regression to include `BRAND`, `TIME_OF_DAY`, `VEHICLE_TYPE_CLEAN`, and `CAUSE`, standard Maximum Likelihood Estimation (`sm.Logit`) encountered a singular matrix / separation issue because **`NISSAN` had `0.000000` fatalities** in the sample. In logistic regression, when a binary dummy perfectly predicts $Y = 0$, its MLE coefficient diverges toward $-\infty$. Below, we show the exact diagnostic steps used to detect and resolve this by dropping `BRAND_NISSAN` from the unpenalized design matrix.

In [ ]:
# ==============================================================================
# EXPERIMENT 2A: TOP 5 BRANDS FATALITY RATE PER 10,000 CRASHES
# ==============================================================================
df_brands = my_data.copy()
df_brands["CLEAN_MAKE"] = df_brands["VEHICLE_MAKE"].astype(str).str.upper().str.split().str[0]
top_brands = ["TOYT", "HOND", "NISS", "FORD", "CHEV"]
df_cars = df_brands[df_brands["CLEAN_MAKE"].isin(top_brands)].copy()
df_cars["IS_KILLED"] = (pd.to_numeric(df_cars["NUMBER OF PERSONS KILLED"], errors="coerce") > 0).astype(int)

print("Crash and Death Counts by Top 5 Brands:")
brand_summary = (
    df_cars.groupby("CLEAN_MAKE")["IS_KILLED"]
    .agg(["count", "sum"])
    .rename(columns={"count": "Total Crashes", "sum": "Fatal Crashes"})
)
brand_summary["Fatal Crashes per 10k"] = (brand_summary["Fatal Crashes"] / brand_summary["Total Crashes"]) * 10000
display(brand_summary.sort_values(by="Fatal Crashes per 10k", ascending=False))

# ==============================================================================
# EXPERIMENT 2B: MULTI-FACTOR LOGIT & PERFECT SEPARATION DIAGNOSTIC (NISSAN = 0)
# ==============================================================================
df_sep = df_raw.copy() if raw_csv_path is not None else my_data.copy()
df_sep["FATAL"] = (pd.to_numeric(df_sep["NUMBER OF PERSONS KILLED"], errors="coerce").fillna(0) > 0).astype(int)

def get_brand_upper(make: str) -> str:
    m = str(make).upper().strip()
    if "FORD" in m: return "FORD"
    if "TOYT" in m or "TOYOTA" in m: return "TOYOTA"
    if "HOND" in m: return "HONDA"
    if "NISS" in m: return "NISSAN"
    if "CHEV" in m: return "CHEVROLET"
    return "OTHER"

df_sep["BRAND"] = df_sep["VEHICLE_MAKE"].apply(get_brand_upper)
df_sep["CRASH_TIME_DT"] = pd.to_datetime(df_sep["CRASH_TIME"], format="%H:%M", errors="coerce")
df_sep["HOUR"] = df_sep["CRASH_TIME_DT"].dt.hour
df_sep["TIME_OF_DAY"] = df_sep["HOUR"].apply(lambda h: "UNKNOWN" if pd.isna(h) else ("DAY" if 6 <= h < 18 else "NIGHT"))

def get_veh_type_broad(vt: str) -> str:
    v = str(vt).upper()
    if any(k in v for k in ["SEDAN", "4 DR", "2 DR", "PASSENGER"]): return "SEDAN"
    if any(k in v for k in ["SUV", "SPORT UTILITY"]): return "SUV"
    if any(k in v for k in ["TRUCK", "BUS", "VAN"]): return "HEAVY"
    return "OTHER_TYPE"

df_sep["VEHICLE_TYPE_CLEAN"] = df_sep["VEHICLE_TYPE"].apply(get_veh_type_broad)

def group_factor_broad(factor: str) -> str:
    f = str(factor).upper()
    if "DISTRACTION" in f or "INATTENTION" in f: return "DISTRACTION"
    if "SPEED" in f: return "SPEED"
    if "ALCOHOL" in f or "DRUG" in f: return "SUBSTANCE"
    if any(k in f for k in ["YIELD", "TRAFFIC CONTROL", "STOP"]): return "TRAFFIC_VIOLATION"
    if "UNSPECIFIED" in f: return "UNSPECIFIED"
    return "OTHER_CAUSE"

df_sep["CAUSE"] = df_sep["CONTRIBUTING_FACTOR_1"].apply(group_factor_broad)
df_model_sep = df_sep[df_sep["TIME_OF_DAY"] != "UNKNOWN"].copy()

print("\nFatality Rate (%) by Brand:")
print(df_model_sep.groupby("BRAND")["FATAL"].mean() * 100)

brand_dummies = pd.get_dummies(df_model_sep["BRAND"], prefix="BRAND").drop(columns=["BRAND_OTHER"], errors="ignore")
time_dummies = pd.get_dummies(df_model_sep["TIME_OF_DAY"], prefix="TIME").drop(columns=["TIME_DAY"], errors="ignore")
type_dummies = pd.get_dummies(df_model_sep["VEHICLE_TYPE_CLEAN"], prefix="TYPE").drop(columns=["TYPE_OTHER_TYPE"], errors="ignore")
cause_dummies = pd.get_dummies(df_model_sep["CAUSE"], prefix="CAUSE").drop(columns=["CAUSE_UNSPECIFIED"], errors="ignore")

X_sep = pd.concat([brand_dummies, time_dummies, type_dummies, cause_dummies], axis=1).astype(float)
X_sep = sm.add_constant(X_sep)
y_sep = df_model_sep["FATAL"].astype(float)

# Drop BRAND_NISSAN because 0 fatalities causes perfect separation in unpenalized Logit
X_clean = X_sep.drop(columns=["BRAND_NISSAN"], errors="ignore")

print("\n--- Summary Statistics for Predictor Variables (X_clean) ---")
display(X_clean.describe().round(4))
print("\n--- Summary Statistics for Target Variable (Y: FATAL) ---")
display(y_sep.describe().round(4))

try:
    logit_sep = sm.Logit(y_sep, X_clean).fit(disp=False, maxiter=100)
    print(logit_sep.summary())
except Exception:
    logit_sep = sm.Logit(y_sep, X_clean).fit_regularized(disp=False)
    print(logit_sep.summary())

## Phase 5: Experiment 3 — Modular Block-Wise Hypothesis Testing (Sub-Models 1–4)
Before combining all predictors into our unified Master Logit Model, we tested `FORD` alongside each individual hypothesis block on the cleaned dataset ($N = 39,793$):
* **Model 1 (Gender):** `FATAL ~ FORD + MALE`
* **Model 2 (Time of Day):** `FATAL ~ FORD + AFTERNOON + NIGHT` (Baseline = `MORNING` / `PEAK`)
* **Model 3 (Vehicle Type):** `FATAL ~ FORD + TRUCK`
* **Model 4 (Seasonality):** `FATAL ~ FORD + SPRING + SUMMER + WINTER` (Baseline = `FALL`)

In [ ]:
# ==============================================================================
# EXPERIMENT 3: ISOLATED HYPOTHESIS SUB-MODELS (MODELS 1 - 4)
# ==============================================================================
df_sub = my_data.copy()
df_sub["FATAL"] = (pd.to_numeric(df_sub["NUMBER OF PERSONS KILLED"], errors="coerce").fillna(0) > 0).astype(int)
df_sub["FORD"] = df_sub["VEHICLE_MAKE"].astype(str).str.upper().str.contains("FORD").astype(int)

# --- Model 1: Gender Hypothesis ---
df_m1 = df_sub[["FATAL", "FORD", "DRIVER_SEX"]].dropna()
df_m1 = df_m1[df_m1["DRIVER_SEX"].isin(["M", "F"])].copy()
df_m1["MALE"] = (df_m1["DRIVER_SEX"] == "M").astype(int)

print("--- MODEL 1: GENDER ---")
m1 = smf.logit("FATAL ~ FORD + MALE", data=df_m1).fit(disp=False)
print(m1.summary())

# --- Model 2: Time of Day Hypothesis ---
df_sub["CRASH_TIME_DT"] = pd.to_datetime(df_sub["CRASH_TIME"], format="%H:%M", errors="coerce")
df_sub["HOUR"] = df_sub["CRASH_TIME_DT"].dt.hour
df_sub["MORNING"] = ((df_sub["HOUR"] >= 6) & (df_sub["HOUR"] < 12)).astype(int)
df_sub["AFTERNOON"] = ((df_sub["HOUR"] >= 12) & (df_sub["HOUR"] < 16)).astype(int)
df_sub["PEAK"] = ((df_sub["HOUR"] >= 16) & (df_sub["HOUR"] < 20)).astype(int)
df_sub["NIGHT"] = ((df_sub["HOUR"] >= 20) | (df_sub["HOUR"] < 6)).astype(int)

df_m2 = df_sub[["FATAL", "FORD", "AFTERNOON", "NIGHT"]].dropna()
print("\n--- MODEL 2: TIME OF DAY ---")
m2 = smf.logit("FATAL ~ FORD + AFTERNOON + NIGHT", data=df_m2).fit(disp=False)
print(m2.summary())

# --- Model 3: Vehicle Type (Trucks/SUVs) Hypothesis ---
df_sub["TRUCK"] = df_sub["VEHICLE_TYPE"].astype(str).str.upper().str.contains("TRUCK|SUV|VAN|UTILITY").astype(int)
df_m3 = df_sub[["FATAL", "FORD", "TRUCK"]].dropna()
print("\n--- MODEL 3: VEHICLE TYPE ---")
m3 = smf.logit("FATAL ~ FORD + TRUCK", data=df_m3).fit(disp=False)
print(m3.summary())

# --- Model 4: Seasonality Hypothesis (Baseline = FALL) ---
df_sub["CRASH_DATE_DT"] = pd.to_datetime(df_sub["CRASH_DATE"], errors="coerce")
df_sub["MONTH"] = df_sub["CRASH_DATE_DT"].dt.month
df_sub["WINTER"] = df_sub["MONTH"].isin([12, 1, 2]).astype(int)
df_sub["SPRING"] = df_sub["MONTH"].isin([3, 4, 5]).astype(int)
df_sub["SUMMER"] = df_sub["MONTH"].isin([6, 7, 8]).astype(int)
df_sub["FALL"] = df_sub["MONTH"].isin([9, 10, 11]).astype(int)

df_m4 = df_sub[["FATAL", "FORD", "WINTER", "SPRING", "SUMMER", "FALL"]].dropna()
print("\n--- MODEL 4: SEASONALITY (Baseline = FALL) ---")
m4 = smf.logit("FATAL ~ FORD + SPRING + SUMMER + WINTER", data=df_m4).fit(disp=False)
print(m4.summary())

## Phase 6: Experiment 4 — Outlier Sanitization, Final Master Logit Model & Male Subsample
Here we apply the outlier filter discovered in Phase 2 (`1900 <= VEHICLE_YEAR <= 2026` and `VEHICLE_OCCUPANTS < 100`, recalculating `VEHICLE_AGE = 2026 - VEHICLE_YEAR`, yielding **$N = 39,689$** valid observations).

We then demonstrate the critical modeling breakthrough of our project:
1. **Intermediate 9-Predictor Model (Omitting Vehicle Body Types):** Yields an insignificant overall model (LLR $p = 0.1168$, Pseudo $R^2 = 0.03507$).
2. **Final 12-Predictor Master Logit Model (Adding `TYPE_TRUCK`, `TYPE_SUV`, `TYPE_SEDAN`):** Jumps Pseudo $R^2$ to **`0.08973`** and achieves high global statistical significance (**LLR $p = 0.0002980$**):
$$\log\left(\frac{P(\text{FATAL}=1)}{1 - P(\text{FATAL}=1)}\right) = \beta_0 + \beta_1(\text{FORD}) + \beta_2(\text{MALE}) + \beta_3(\text{VEHICLE\_AGE}) + \beta_4(\text{TYPE\_TRUCK}) + \beta_5(\text{TYPE\_SUV}) + \beta_6(\text{TYPE\_SEDAN}) + \beta_7(\text{AFTERNOON}) + \beta_8(\text{PEAK}) + \beta_9(\text{NIGHT}) + \beta_{10}(\text{SPRING}) + \beta_{11}(\text{SUMMER}) + \beta_{12}(\text{WINTER})$$
3. **Subsample Sensitivity Analysis (Male Drivers Only, $N = 28,585$):** Confirms that even within the male driver population, `FORD` is not a statistically significant predictor of fatalities ($p = 0.467$).

In [ ]:
# ==============================================================================
# EXPERIMENT 4: OUTLIER FILTER, INTERMEDIATE LOGIT, MASTER LOGIT & MALE SUBSAMPLE
# ==============================================================================
df_master = my_data.copy()
df_master["VEHICLE_YEAR"] = pd.to_numeric(df_master["VEHICLE_YEAR"], errors="coerce")
df_master["VEHICLE_OCCUPANTS"] = pd.to_numeric(df_master["VEHICLE_OCCUPANTS"], errors="coerce")

# 1. Apply Outlier Filter (Realistic Vehicle Years [1900, 2026] and Occupancy < 100)
df_master = df_master[(df_master["VEHICLE_YEAR"] >= 1900) & (df_master["VEHICLE_YEAR"] <= 2026)].copy()
df_master = df_master[df_master["VEHICLE_OCCUPANTS"] < 100].copy()
df_master["VEHICLE_AGE"] = 2026 - df_master["VEHICLE_YEAR"]

# 2. Engineer Target & Predictors
df_master["FATAL"] = (pd.to_numeric(df_master["NUMBER OF PERSONS KILLED"], errors="coerce").fillna(0) > 0).astype(int)
df_master["FORD"] = df_master["VEHICLE_MAKE"].astype(str).str.upper().str.contains("FORD").astype(int)

df_master = df_master[df_master["DRIVER_SEX"].isin(["M", "F"])].copy()
df_master["MALE"] = (df_master["DRIVER_SEX"] == "M").astype(int)

def classify_veh_type(vt: str) -> str:
    v = str(vt).upper()
    if any(k in v for k in ["SEDAN", "4 DR", "2 DR", "PASSENGER"]):
        return "SEDAN"
    if any(k in v for k in ["SUV", "SPORT UTILITY", "SPORT"]):
        return "SUV"
    if any(k in v for k in ["TRUCK", "BUS", "VAN", "UTILITY"]):
        return "TRUCK"
    return "OTHER_TYPE"

df_master["VEHICLE_TYPE_CLEAN"] = df_master["VEHICLE_TYPE"].apply(classify_veh_type)
df_master["TYPE_TRUCK"] = (df_master["VEHICLE_TYPE_CLEAN"] == "TRUCK").astype(int)
df_master["TYPE_SUV"] = (df_master["VEHICLE_TYPE_CLEAN"] == "SUV").astype(int)
df_master["TYPE_SEDAN"] = (df_master["VEHICLE_TYPE_CLEAN"] == "SEDAN").astype(int)
df_master["TRUCK"] = df_master["VEHICLE_TYPE"].astype(str).str.upper().str.contains("TRUCK|SUV|VAN|UTILITY").astype(int)

df_master["CRASH_TIME_DT"] = pd.to_datetime(df_master["CRASH_TIME"], format="%H:%M", errors="coerce")
df_master["HOUR"] = df_master["CRASH_TIME_DT"].dt.hour
df_master["MORNING"] = ((df_master["HOUR"] >= 6) & (df_master["HOUR"] < 12)).astype(int)
df_master["AFTERNOON"] = ((df_master["HOUR"] >= 12) & (df_master["HOUR"] < 16)).astype(int)
df_master["PEAK"] = ((df_master["HOUR"] >= 16) & (df_master["HOUR"] < 20)).astype(int)
df_master["NIGHT"] = ((df_master["HOUR"] >= 20) | (df_master["HOUR"] < 6)).astype(int)

df_master["CRASH_DATE_DT"] = pd.to_datetime(df_master["CRASH_DATE"], errors="coerce")
df_master["MONTH"] = df_master["CRASH_DATE_DT"].dt.month
df_master["WINTER"] = df_master["MONTH"].isin([12, 1, 2]).astype(int)
df_master["SPRING"] = df_master["MONTH"].isin([3, 4, 5]).astype(int)
df_master["SUMMER"] = df_master["MONTH"].isin([6, 7, 8]).astype(int)
df_master["FALL"] = df_master["MONTH"].isin([9, 10, 11]).astype(int)

print("--- Sanitized Dataset Summary Statistics (N = 39,689) ---")
display(df_master.describe())

# 3. Intermediate 9-Predictor Logit Model (Before Adding Vehicle Body Type Dummies)
formula_9pred = "FATAL ~ FORD + MALE + VEHICLE_AGE + AFTERNOON + PEAK + NIGHT + SPRING + SUMMER + WINTER"
m_intermediate = smf.logit(formula_9pred, data=df_master).fit(disp=False)
print("\n--- INTERMEDIATE LOGIT MODEL (9 PREDICTORS — WITHOUT VEHICLE BODY TYPES) ---")
print(m_intermediate.summary())

# 4. Final 12-Predictor Master Logit Model
formula_master = (
    "FATAL ~ FORD + MALE + VEHICLE_AGE + TYPE_TRUCK + TYPE_SUV + TYPE_SEDAN "
    "+ AFTERNOON + PEAK + NIGHT + SPRING + SUMMER + WINTER"
)
m_master = smf.logit(formula_master, data=df_master).fit(disp=False)
print("\n--- FINAL MASTER LOGIT MODEL (12 PREDICTORS) ---")
print(m_master.summary())

# 5. Subsample Regression: Male Drivers Only
df_males_only = df_master[df_master["MALE"] == 1].copy()
formula_males = "FATAL ~ FORD + TRUCK + SPRING + SUMMER + FALL + PEAK + MORNING + AFTERNOON + VEHICLE_AGE"
m_males = smf.logit(formula_males, data=df_males_only).fit(disp=False)
print("\n--- SUBSAMPLE LOGIT MODEL: MALE DRIVERS ONLY ---")
print(m_males.summary())

## Phase 7: Experiment 5 — Scikit-Learn Logistic Regression Pipeline & Regularization Benchmark
In the unpenalized `statsmodels` Master Logit above, notice the warning at the bottom of the output:
> *`Possibly complete quasi-separation: A fraction 0.21 of observations can be perfectly predicted.`*

Because `SPRING` has almost no fatal collisions in the sample, unpenalized Maximum Likelihood Estimation assigns `SPRING` an unstable coefficient (`-4.4031`) with an inflated standard error (`26.190`).

To showcase modern Python machine learning conventions, we implement a **`scikit-learn` Logistic Regression workflow**:
1. **Direct $L_2$ (Ridge) Penalized Comparison:** Demonstrates how $L_2$ regularization shrinks the unstable `SPRING` coefficient into a well-behaved range while preserving the exact directional insights on `VEHICLE_AGE`, `NIGHT`, `TYPE_SUV`, and `TYPE_SEDAN`.
2. **Production ML Pipeline (`ColumnTransformer` + `StandardScaler` + `StratifiedKFold`):** Evaluates out-of-sample discriminative performance via 5-fold cross-validated **ROC-AUC**.

In [ ]:
# ==============================================================================
# EXPERIMENT 5: SCIKIT-LEARN LOGISTIC REGRESSION & REGULARIZATION BENCHMARK
# ==============================================================================
MASTER_FEATURES = [
    "FORD", "MALE", "VEHICLE_AGE", "TYPE_TRUCK", "TYPE_SUV", "TYPE_SEDAN",
    "AFTERNOON", "PEAK", "NIGHT", "SPRING", "SUMMER", "WINTER"
]

df_sk = df_master.dropna(subset=["FATAL"] + MASTER_FEATURES).copy()
X_sk = df_sk[MASTER_FEATURES]
y_sk = df_sk["FATAL"].values

# 1. Fit L2-Penalized Logistic Regression on raw feature scales for direct comparison with Statsmodels
sk_logit_unscaled = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000)
sk_logit_unscaled.fit(X_sk, y_sk)

comparison_table = pd.DataFrame({
    "Statsmodels_MLE_Coef": m_master.params.drop("Intercept"),
    "Statsmodels_Std_Err": m_master.bse.drop("Intercept"),
    "Statsmodels_P_Value": m_master.pvalues.drop("Intercept"),
    "Sklearn_L2_Coef": pd.Series(sk_logit_unscaled.coef_[0], index=MASTER_FEATURES),
    "Sklearn_Odds_Ratio": np.exp(pd.Series(sk_logit_unscaled.coef_[0], index=MASTER_FEATURES)),
}).round(4)

print("--- Econometric MLE (Statsmodels) vs. L2-Regularized Logit (Scikit-Learn) ---")
display(comparison_table)

# 2. Build a full Scikit-Learn Preprocessing + Balanced Logistic Regression Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["VEHICLE_AGE"]),
        ("bin", "passthrough", [c for c in MASTER_FEATURES if c != "VEHICLE_AGE"]),
    ]
)

ml_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(C=1.0, class_weight="balanced", solver="lbfgs", max_iter=1000, random_state=42)),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(ml_pipeline, X_sk, y_sk, cv=cv, scoring="roc_auc")
print(f"\n5-Fold Stratified Cross-Validation ROC-AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})")

X_train, X_test, y_train, y_test = train_test_split(
    X_sk, y_sk, test_size=0.25, stratify=y_sk, random_state=42
)
ml_pipeline.fit(X_train, y_train)
y_pred = ml_pipeline.predict(X_test)
y_prob = ml_pipeline.predict_proba(X_test)[:, 1]

print(f"Holdout Test Set ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print("\nHoldout Classification Report (with class_weight='balanced'):")
print(classification_report(y_test, y_pred, target_names=["Non-Fatal (0)", "Fatal (1)"]))

## Phase 8: Visual Analytics Suite (Interactive Plotly & Publication Seaborn Figures)
Finally, we generate all visual proofs supporting our four hypotheses:
1. **Master Logit Coefficient Forest Plot:** Highlights statistically significant predictors ($p < 0.05$) and their 95% confidence intervals.
2. **Heatmaps of Fatality Rate (%):** Cross-tabulates severity across `Vehicle Type x Time of Day`, `Vehicle Type x Crash Cause`, and `Month x Time of Day`.
3. **Impact of Vehicle Age on Fatality Rate (Bubble Scatter + OLS Trendline):** Visually confirms our positive `VEHICLE_AGE` Logit coefficient (`+0.0552`), demonstrating that structural aging and older safety eras increase crash lethality regardless of manufacturer.

In [ ]:
# ==============================================================================
# PHASE 8: PUBLICATION VISUALIZATIONS (PLOTLY & SEABORN)
# ==============================================================================
def group_cause_clean(factor: str) -> str:
    f = str(factor).upper()
    if "DISTRACTION" in f or "INATTENTION" in f: return "DISTRACTION"
    if "SPEED" in f: return "SPEED"
    if "ALCOHOL" in f or "DRUG" in f: return "SUBSTANCE"
    if any(k in f for k in ["YIELD", "TRAFFIC CONTROL", "STOP"]): return "TRAFFIC_VIOLATION"
    if "UNSPECIFIED" in f: return "UNSPECIFIED"
    return "OTHER_CAUSE"

df_master["CAUSE_CLEAN"] = df_master["CONTRIBUTING_FACTOR_1"].apply(group_cause_clean)

def get_time_label_clean(row: pd.Series) -> str:
    if row["MORNING"] == 1: return "Morning"
    if row["AFTERNOON"] == 1: return "Afternoon"
    if row["PEAK"] == 1: return "Peak"
    if row["NIGHT"] == 1: return "Night"
    return "Unknown"

df_master["TIME_LABEL"] = df_master.apply(get_time_label_clean, axis=1)

# ------------------------------------------------------------------------------
# 1. Forest Plot of Master Logit Coefficients (Interactive Plotly)
# ------------------------------------------------------------------------------
conf = m_master.conf_int()
conf.columns = ["CI_Lower", "CI_Upper"]
conf["Coefficient"] = m_master.params
conf["P_Value"] = m_master.pvalues
conf = conf.drop("Intercept", errors="ignore").sort_values(by="Coefficient")

# Omit SPRING from visual whisker plot if quasi-separation CI is +/- 55
conf_vis = conf[conf.index != "SPRING"].copy()

fig_forest = go.Figure()
fig_forest.add_trace(
    go.Scatter(
        x=conf_vis["Coefficient"],
        y=conf_vis.index,
        mode="markers",
        name="Coefficient",
        marker=dict(
            size=12,
            color=np.where(conf_vis["P_Value"] < 0.05, "red", "gray"),
        ),
        error_x=dict(
            type="data",
            symmetric=False,
            array=conf_vis["CI_Upper"] - conf_vis["Coefficient"],
            arrayminus=conf_vis["Coefficient"] - conf_vis["CI_Lower"],
            color="black",
            thickness=2,
        ),
    )
)
fig_forest.add_vline(x=0, line_dash="dash", line_color="black")
fig_forest.update_layout(
    title="Master Logit Model Coefficients (Forest Plot)<br><sup>Red = Significant (p < 0.05) | Gray = Insignificant | Whiskers = 95% CI</sup>",
    xaxis_title="Log-Odds Effect (Right of 0 = Increases Fatality, Left = Decreases)",
    yaxis_title="Predictor Variable",
    template="plotly_white",
    height=550,
)
fig_forest.write_html(OUTPUT_DIR / "forest_plot.html")
fig_forest.show()

# ------------------------------------------------------------------------------
# 2. Heatmap of Fatality Rate (%) by Vehicle Type and Crash Cause
# ------------------------------------------------------------------------------
heatmap_cause = df_master.groupby(["VEHICLE_TYPE_CLEAN", "CAUSE_CLEAN"])["FATAL"].mean().unstack() * 100
fig_heat_cause = px.imshow(
    heatmap_cause,
    text_auto=".3f",
    title="Fatality Rate (%) by Vehicle Type and Crash Cause",
    color_continuous_scale="Reds",
    labels=dict(x="Crash Cause", y="Vehicle Type", color="Fatality %"),
)
fig_heat_cause.update_layout(height=480, template="plotly_white")
fig_heat_cause.write_html(OUTPUT_DIR / "heatmap_cause_vs_type.html")
fig_heat_cause.show()

# ------------------------------------------------------------------------------
# 3. Static Seaborn Bubble Scatter Plot + OLS Trendline: Vehicle Age vs Fatality
# ------------------------------------------------------------------------------
df_age = df_master[(df_master["VEHICLE_AGE"] >= 0) & (df_master["VEHICLE_AGE"] <= 40)].copy()
age_stats = (
    df_age.groupby("VEHICLE_AGE")
    .agg(FATAL_RATE=("FATAL", "mean"), CRASH_COUNT=("FATAL", "count"))
    .reset_index()
)
age_stats["FATAL_RATE_PCT"] = age_stats["FATAL_RATE"] * 100

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")
sns.regplot(
    data=age_stats,
    x="VEHICLE_AGE",
    y="FATAL_RATE_PCT",
    scatter=False,
    color="darkred",
    line_kws={"linewidth": 2, "linestyle": "--"},
)
sns.scatterplot(
    data=age_stats,
    x="VEHICLE_AGE",
    y="FATAL_RATE_PCT",
    size="CRASH_COUNT",
    sizes=(50, 1500),
    alpha=0.7,
    color="#e74c3c",
    edgecolor="black",
    legend=False,
)
plt.title("Impact of Vehicle Age on Fatality Rate", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Vehicle Age (Years)", fontsize=12, fontweight="bold")
plt.ylabel("Fatality Rate (%)", fontsize=12, fontweight="bold")
plt.text(
    0.98,
    0.05,
    "* Bubble size indicates total crashes recorded for that age group",
    transform=plt.gca().transAxes,
    fontsize=11,
    ha="right",
    style="italic",
    color="dimgrey",
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "static_vehicle_age_fatality.png", dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------------------------
# 4. Static Seaborn Calendar Heatmap: Fatality Rate (%) by Month and Time of Day
# ------------------------------------------------------------------------------
df_cal = df_master.dropna(subset=["MONTH"]).copy()
df_cal["MONTH_NAME"] = df_cal["MONTH"].apply(lambda x: calendar.month_abbr[int(x)] if pd.notna(x) else "Unknown")
df_cal = df_cal[(df_cal["TIME_LABEL"] != "Unknown") & (df_cal["MONTH_NAME"] != "Unknown")]

cal_matrix = df_cal.groupby(["MONTH_NAME", "TIME_LABEL"])["FATAL"].mean().unstack() * 100
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
time_order = ["Morning", "Afternoon", "Peak", "Night"]
cal_matrix = cal_matrix.reindex(index=month_order, columns=time_order)

plt.figure(figsize=(12, 8))
sns.set_theme(style="white")
sns.heatmap(
    cal_matrix,
    annot=True,
    fmt=".3f",
    cmap="Reds",
    linewidths=0.5,
    cbar_kws={"label": "Fatality Rate (%)"},
)
plt.title("Fatality Rate (%) by Month and Time of Day", fontsize=16, fontweight="bold", pad=20)
plt.xlabel("Time of Day", fontsize=14, fontweight="bold")
plt.ylabel("Month", fontsize=14, fontweight="bold")
plt.xticks(rotation=0, fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_month_vs_time.png", dpi=300, bbox_inches="tight")
plt.show()